# 16 — Dựng artifact D6: structured output (máy local)

**Phạm vi**

Xuất năm bảng quan hệ độc lập công cụ vào `06_output_data_model/`: `reviews`, `topics`, `requirements`, `requirement_evidence`,
`model_comparison`. Mọi giá trị lấy từ kết quả đã khóa trong `05_experiments/` (GĐ10–GĐ15); không dự đoán hay tính lại. Logic ở
`src/output_layer/core.py`.

**Văn bản review**

`reviews.csv` chứa văn bản review (bản interim đã scrub) để BA đọc evidence, nên không commit (`.gitignore`); chạy lại notebook này để
tạo lại. Bốn bảng còn lại không có văn bản review.

In [1]:
import os, sys, json, datetime, subprocess
import pandas as pd, yaml
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.utils.display import show as show_table
from src.classification.full_inference import load_corpus
from src.output_layer import core as O
RQ1 = os.path.join(ROOT, '05_experiments', 'rq1_classification')
RQ2 = os.path.join(ROOT, '05_experiments', 'rq2_topics')
RQ3 = os.path.join(ROOT, '05_experiments', 'rq3_requirements')
OUT = os.path.join(ROOT, '06_output_data_model')
COMMIT = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Ngày chạy:', datetime.date.today().strftime('%d/%m/%Y'), '| commit:', COMMIT)

Ngày chạy: 03/11/2026 | commit: 57f9591c2b78d28bd201364a7d283daf10ec459f


## Đầu vào

**Nguồn của từng bảng**

`reviews`: corpus Shopee interim (văn bản, ngày, phiên bản, số sao) + dự đoán GĐ11 (`gd11_corpus_predictions.csv`) + cụm GĐ12
(`review_topic_map.csv`) + trạng thái evidence GĐ14 (`gd14_evidence_pool.csv`) + trạng thái temporal của unit (GĐ14) hoặc của cụm (GĐ13).
`topics`: `topics.csv`, manual purity, trạng thái temporal cấp cụm, cách GĐ14 xử lý cụm (config frozen). `requirements` và
`requirement_evidence`: GĐ14, kèm lỗi QUS của GĐ15 và điểm chấm tay nếu đã có. `model_comparison`: các bảng GĐ10 trên Hold-out (mở một lần).

In [2]:
CORPUS = load_corpus(os.path.join(ROOT, '01_data', 'interim', 'shopee_clean_T0_2026-10-06.jsonl'))
PREDS = pd.read_csv(os.path.join(RQ2, 'gd11_corpus_predictions.csv'))
assert set(CORPUS['review_id']) == set(PREDS['review_id']), 'corpus và dự đoán GĐ11 không khớp'
MAP = pd.read_csv(os.path.join(RQ2, 'review_topic_map.csv'))
TOPICS = pd.read_csv(os.path.join(RQ2, 'topics.csv'), encoding='utf-8-sig')
PURITY = pd.read_csv(os.path.join(RQ2, 'gd12_manual_purity.csv'), encoding='utf-8-sig')
TGROUPS = pd.read_csv(os.path.join(RQ2, 'gd13_temporal_groups.csv'))
POOL = pd.read_csv(os.path.join(RQ3, 'gd14_evidence_pool.csv'), dtype={'unit_id': str})
UNITS = pd.read_csv(os.path.join(RQ3, 'gd14_generation_units.csv'), encoding='utf-8-sig', dtype={'unit_id': str})
REQ = pd.read_csv(os.path.join(RQ3, 'requirements.csv'), encoding='utf-8-sig')
EV = pd.read_csv(os.path.join(RQ3, 'requirement_evidence.csv'))
QUS = pd.read_csv(os.path.join(RQ3, 'gd15_qus_defects.csv'))
CFG = yaml.safe_load(open(os.path.join(ROOT, '03_configs', 'frozen', 'rq3_generation_config.yaml'), encoding='utf-8'))['rq3_generation']
HP = os.path.join(RQ3, 'gd15_human_per_requirement.csv')
HUMAN = pd.read_csv(HP) if os.path.exists(HP) else None
HUMAN_EV = None
if HUMAN is not None:
    from src.evaluation import core as E
    sheets = E.read_sheets(os.path.join(ROOT, '07_human_eval', 'rq3_requirements'))
    HUMAN_EV = sheets[list(sheets)[0]][1][['requirement_id', 'review_id']].assign(
        supports=sum(s[1]['supports'].to_numpy(float) for s in sheets.values()) / len(sheets))
RQ1T = {k: pd.read_csv(os.path.join(RQ1, f), encoding='utf-8-sig') for k, f in
        [('tier1', 'gd10_tier1_holdout.csv'), ('tier2', 'gd10_tier2_holdout.csv'), ('e2e', 'gd10_e2e_holdout.csv'),
         ('e2e_lazada', 'gd10_e2e_lazada.csv'), ('cost', 'gd10_cost_summary.csv')]}
print('Corpus:', len(CORPUS), '| actionable đã gom cụm:', len(MAP), '| requirement:', len(REQ),
      '| điểm chấm tay:', 'có' if HUMAN is not None else 'chưa có (cột eval_* sẽ thêm khi chạy lại sau GĐ15)')

Corpus: 56050 | actionable đã gom cụm: 10897 | requirement: 16 | điểm chấm tay: có


## Dựng năm bảng và kiểm tra khóa

In [3]:
TABLES = {}
TABLES['reviews'] = O.reviews_table(CORPUS, PREDS, MAP, POOL, UNITS, TGROUPS)
TABLES['topics'] = O.topics_table(TOPICS, PURITY, TGROUPS, CFG)
TABLES['requirements'] = O.requirements_table(REQ, QUS, HUMAN)
TABLES['requirement_evidence'] = O.evidence_table(EV, TABLES['reviews'], HUMAN_EV)
TABLES['model_comparison'] = O.model_comparison(RQ1T)
KEYS = O.check_keys(TABLES)
show_table(KEYS, 'Ràng buộc khóa giữa các bảng')
assert KEYS['đạt'].all(), 'vi phạm ràng buộc khóa — dừng'
show_table(pd.DataFrame([{'bảng': k, 'số dòng': len(v), 'số cột': v.shape[1], 'có văn bản review': 'text' in v.columns}
                         for k, v in TABLES.items()]), 'Năm bảng D6')
show_table(TABLES['reviews']['evidence_status'].replace('', '(không actionable)').value_counts()
           .rename_axis('evidence_status').reset_index(name='số review'), 'reviews.evidence_status')
show_table(TABLES['topics'].drop(columns=['top_keywords']), 'topics', decimals=2, percent=['frequency'])
show_table(TABLES['model_comparison'].drop(columns=['per_class_f1']), 'model_comparison', decimals=2)

ràng buộc,đạt
reviews.review_id duy nhất,True
reviews.topic_id → topics.topic_id,True
requirements.topic_id → topics.topic_id,True
requirement_evidence.requirement_id → requirements,True
requirement_evidence.review_id → reviews,True
mọi requirement có evidence,True


bảng,số dòng,số cột,có văn bản review
reviews,56.050,14,True
topics,18,13,False
requirements,16,22,False
requirement_evidence,93,7,False
model_comparison,10,14,False


evidence_status,số review
(không actionable),45.153
evidence_pool,5.571
ba_manual,2.667
outlier,1.298
out_of_scope_app,1.168
type_mismatch,188
temporal_insufficient,5


topic_id,label,label_en,dominant_type,type_purity,manual_purity,size,frequency,scope,temporal_status,generation_step,flag_ba
-1,outlier,outlier,,"0,49",,1.298,"11,9%",,,outlier,False
0,Dịch vụ giao hàng kém chất lượng,Poor delivery service,mixed,"0,56","0,90",1.232,"11,3%",ngoài_app,RECURRING,scope_filter,False
1,Lỗi tài khoản và thanh toán,Account and payment errors,BUG,"0,79","1,00",1.098,"10,1%",app,RECURRING,normal,False
2,Quảng cáo gây phiền phức,Annoying advertisements,mixed,"0,47","0,20",1.322,"12,1%",app,RECURRING,ba_manual,True
3,Quảng cáo xuất hiện liên tục,Continuous advertisement display,UX,"0,98","0,70",1.287,"11,8%",app,RECURRING,normal,False
4,Video tự phát âm thanh phiền phức,Annoying autoplay videos,mixed,"0,50","0,20",573,"5,3%",app,RECURRING,ba_manual,True
5,Quảng cáo rác và khó chịu,Annoying junk advertisements,UX,"0,70","0,70",635,"5,8%",app,RECURRING,normal,False
6,Ứng dụng lag và chậm,Laggy and slow app,BUG,"0,79","0,60",435,"4,0%",app,RECURRING,normal,False
7,Lỗi trong giỏ hàng và mua sắm,Errors in cart and shopping,mixed,"0,47","0,50",517,"4,7%",app,RECURRING,q14_split,False
8,Chất lượng sản phẩm và giao hàng,Product quality and delivery,FEATURE,"0,78","0,55",447,"4,1%",ngoài_app,RECURRING,scope_filter,False


strategy,tier,metric,macro_f1,macro_f1_sd,actionable_recall,actionable_f1,time_per_1k,api_cost,train_time,hardware,external_test_f1,n_runs
ViSoBERT,tier1,Macro-F1 (2 lớp),"78,90","0,60","69,80","65,90","1,55","0,00","42,70",GPU tối đa 2.51 GB,,3
Bi-LSTM + fastText,tier1,Macro-F1 (2 lớp),"72,60","5,00","84,00","59,70","0,11","0,00","6,40",GPU tối đa 0.62 GB,,3
LLM few-shot,tier1,Macro-F1 (2 lớp),"88,70",,"86,90","81,70","1.103,50","0,21","0,00",không cần GPU (API),,1
ViSoBERT,tier2,Macro-F1 (3 lớp),"76,50","1,60",,,"1,55","0,00","42,70",GPU tối đa 2.51 GB,,3
Bi-LSTM + fastText,tier2,Macro-F1 (3 lớp),"69,60","1,10",,,"0,11","0,00","6,40",GPU tối đa 0.62 GB,,3
LLM few-shot,tier2,Macro-F1 (3 lớp),"89,70",,,,"1.103,50","0,21","0,00",không cần GPU (API),,1
ViSoBERT,end_to_end,Macro-F1 (4 lớp),"62,10","0,70",,,"1,55","0,00","42,70",GPU tối đa 2.51 GB,"56,00",3
Bi-LSTM + fastText,end_to_end,Macro-F1 (4 lớp),"53,50","3,70",,,"0,11","0,00","6,40",GPU tối đa 0.62 GB,"47,10",3
LLM few-shot,end_to_end,Macro-F1 (4 lớp),"81,30",,,,"1.103,50","0,21","0,00",không cần GPU (API),"60,50",1
Baseline phẳng ViSoBERT,flat_baseline_reference,Macro-F1 (4 lớp),"66,80","1,50",,,"1,39","0,00","39,70",GPU tối đa 2.50 GB,,3


## Xuất

CSV UTF-8 có BOM (mở thẳng bằng Excel), dấu thập phân "."; cùng `data_dictionary.csv` mô tả từng cột. Power BI hoặc công cụ BI khác
đọc trực tiếp năm bảng này (`06_output_data_model/powerbi/README.md`).

In [4]:
DESC = {
    'reviews': {'review_id': 'khóa', 'source_app': 'ứng dụng nguồn', 'review_date': 'thời điểm review (ISO, +07:00)',
                'app_version': 'phiên bản app (có thể trống)', 'rating': 'số sao', 'text': 'văn bản review đã làm sạch (interim, đã scrub)',
                'split': 'train/validation/holdout/focused/none', 'p_ACTIONABLE': 'xác suất ACTIONABLE (ViSoBERT tầng 1)',
                'tier1_label': 'ACTIONABLE / NOT_ACTIONABLE (ngưỡng frozen)', 'tier2_label': 'BUG/FEATURE/UX (chỉ review ACTIONABLE)',
                'topic_id': 'cụm GĐ12 (-1 = outlier; trống = không actionable)', 'unit_id': 'unit sinh requirement GĐ14',
                'evidence_status': 'evidence_pool / ba_manual / out_of_scope_app / type_mismatch / mixed_origin_below_q14 / outlier / temporal_insufficient',
                'temporal_status': 'trạng thái temporal của unit hoặc cụm'},
    'topics': {'topic_id': 'khóa', 'label': 'tên cụm tiếng Việt (LLM)', 'label_en': 'tên cụm tiếng Anh', 'dominant_type': 'loại chủ đạo (≥ 60%) hoặc mixed',
               'type_purity': 'tỷ lệ loại đông nhất', 'manual_purity': 'purity chấm tay (2 người, 10 review)', 'size': 'số review',
               'frequency': 'size / tổng actionable', 'top_keywords': 'từ khóa c-TF-IDF', 'scope': 'app / ngoài_app (LLM đặt tên)',
               'temporal_status': 'trạng thái temporal cấp cụm (GĐ13)', 'generation_step': 'cách GĐ14 xử lý (amendment v2.6)',
               'flag_ba': 'cụm chuyển BA xem thủ công'},
    'requirements': {'requirement_id': 'khóa', 'topic_id': 'cụm nguồn', 'unit_id': 'unit (cụm hoặc cụm × loại)', 'type': 'BUG/FEATURE/UX',
                     'title_vi': 'tên requirement tiếng Việt', 'problem_vi': 'vấn đề chung của review được trích (LLM viết)',
                     'user_story': 'User Story tiếng Anh', 'acceptance_criteria': 'danh sách AC Given/When/Then (JSON)',
                     'temporal_status': 'trạng thái temporal của unit', 'issue_frequency': 'số review của unit / tổng actionable (xếp ưu tiên)',
                     'mixed_origin_flag': 'unit lấy từ cụm đã lọc hoặc tách loại', 'scope_note': 'ghi chú lọc và trùng chủ đề cho BA',
                     'n_evidence_pool': 'số review trong tập evidence của unit', 'n_evidence_cited': 'số review được trích',
                     'qus_defects': 'số lỗi QUS (GĐ15)', 'status': 'candidate = chờ BA duyệt',
                     'eval_fidelity': 'điểm chấm tay 1–5', 'eval_actionability': 'điểm chấm tay 1–5', 'eval_ac_testability': 'điểm chấm tay 1–5',
                     'eval_hallucination': 'tỷ lệ người chấm thấy chi tiết bịa', 'eval_type_correct': 'tỷ lệ người chấm thấy đúng loại',
                     'eval_evidence_precision': 'tỷ lệ review trích nêu đúng vấn đề'},
    'requirement_evidence': {'requirement_id': 'khóa ngoài → requirements', 'review_id': 'khóa ngoài → reviews', 'evidence_label': 'nhãn E# trong prompt',
                             'centroid_rank': 'thứ hạng gần tâm tập evidence (1 = gần nhất)', 'source_app': 'ứng dụng nguồn',
                             'review_date': 'thời điểm review', 'eval_supports': 'tỷ lệ người chấm thấy review nêu đúng vấn đề'},
    'model_comparison': {'strategy': 'chiến lược', 'tier': 'tier1 / tier2 / end_to_end / flat_baseline_reference', 'metric': 'tên chỉ số chính',
                         'macro_f1': 'chỉ số chính trên Hold-out (%, trung bình seed)', 'macro_f1_sd': 'độ lệch chuẩn qua seed',
                         'per_class_f1': 'F1 từng lớp (JSON, %)', 'actionable_recall': 'Recall ACTIONABLE (tầng 1, %)',
                         'actionable_f1': 'F1 ACTIONABLE (tầng 1, %)', 'time_per_1k': 'giây suy luận / 1.000 review',
                         'api_cost': 'USD API / 1.000 review', 'train_time': 'giây huấn luyện / seed (tầng 1 + tầng 2)',
                         'hardware': 'phần cứng', 'external_test_f1': 'Macro-F1 4 lớp trên Lazada (external test)', 'n_runs': 'số seed'},
}
dd = []
for name, df in TABLES.items():
    df.to_csv(os.path.join(OUT, f'{name}.csv'), index=False, encoding='utf-8-sig')
    dd += [{'table': name, 'column': c, 'dtype': str(df[c].dtype), 'description': DESC[name].get(c, '')} for c in df.columns]
DD = pd.DataFrame(dd)
assert (DD['description'] != '').all(), DD[DD['description'] == '']
DD.to_csv(os.path.join(OUT, 'data_dictionary.csv'), index=False, encoding='utf-8-sig')
INFO = {'run_date': datetime.date.today().isoformat(), 'commit': COMMIT, 'rows': {k: len(v) for k, v in TABLES.items()},
        'human_eval_included': HUMAN is not None, 'keys_ok': bool(KEYS['đạt'].all())}
json.dump(INFO, open(os.path.join(OUT, 'build_info.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('Đã lưu vào 06_output_data_model/:', sorted(os.listdir(OUT)))

Đã lưu vào 06_output_data_model/: ['README.md', 'build_info.json', 'data_dictionary.csv', 'model_comparison.csv', 'powerbi', 'requirement_evidence.csv', 'requirements.csv', 'reviews.csv', 'topics.csv']


**Nhận xét**

D6 nối toàn bộ pipeline thành một mô hình dữ liệu truy vết được: từ một Candidate Requirement đi qua `requirement_evidence` tới từng
review nguồn (văn bản, ngày, phiên bản, nhãn dự đoán), tới cụm và trạng thái temporal; từ một cụm biết được vì sao nó được sinh,
lọc hay chuyển BA (`generation_step`, `evidence_status`). Ràng buộc khóa được kiểm tra trước khi xuất. Bảng `requirements` mang trạng thái
`candidate`: artifact gợi ý, BA quyết định (amendment v2.8). Chạy lại notebook sau khi chấm tay GĐ15 để thêm các cột `eval_*`.